#Environment

In [ ]:
# SPECIFICATION
# Operating System
!printf "%-22s : %s\n" "Operating System" "$(lsb_release -d | awk -F'\t' '{print $2}')"

# Python Version
!printf "%-22s : %s\n" "Python version" "$(python --version 2>&1 | awk '{print $2}')"

# CPU Model
!printf "%-22s : %s\n" "CPU Model" "$(lscpu | grep 'Model name:' | awk -F': ' '{print $2}' | xargs)"

# Memory
!printf "%-22s : %s\n" "Total Memory" "$(free -h | awk '/Mem:/ {print $2}')"
!free -ght

#GPU
!nvidia-smi

Operating System       : Ubuntu 22.04.3 LTS
Python version         : 3.10.12
CPU Model              : Intel(R) Xeon(R) CPU @ 2.20GHz
Total Memory           : 12Gi
               total        used        free      shared  buff/cache   available
Mem:            12Gi       735Mi       8.8Gi       1.0Mi       3.2Gi        11Gi
Swap:             0B          0B          0B
Total:          12Gi       735Mi       8.8Gi
/bin/bash: line 1: nvidia-smi: command not found


#Library

In [ ]:
!pip install pandas numpy nltk sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
#LIBRARY
import re
import csv
import nltk
import random
import numpy as np
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords, words
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
#RANDOM SEED
random_seed = 42

#Dataset

In [ ]:
#DATASET
# Upload file dataset ikn
from google.colab import files
dataset_train = files.upload()
dataset_val = files.upload()
dataset_test = files.upload()

# Membaca file CSV dataset ikn
train_path = '/content/final_ikn_train_602020.csv'
val_path = '/content/final_ikn_val_602020.csv'
test_path = '/content/final_ikn_test_602020.csv'

data_train = pd.read_csv(train_path, engine='python', encoding= 'utf-8')
data_val = pd.read_csv(val_path, engine='python', encoding= 'utf-8')
data_test = pd.read_csv(test_path, engine='python', encoding= 'utf-8')

data_train.head()

,text,label
0,Rakyat ga setuju IKN.yang setuju kang surpe,negative
1,Bisa diselamatkan dengan cara MK mengambulkan ...,negative
2,Sebenarnya menunjukkan kekhawatiran dan kecema...,negative
3,Lhooo ini ada dlm ritual IKN yg dgn Presiden d...,negative
4,Pejabat pemerintah Pusat yang sudah berada di ...,negative


In [ ]:
#TOTAL SENTIMENT COUNT
# Menghitung dan menampilkan jumlah sentimen pada setiap kelas
sentiment_counts_train = data_train['label'].value_counts()
sentiment_counts_val = data_val['label'].value_counts()
sentiment_counts_test = data_test['label'].value_counts()

sentiment_counts_train

,count
label,
negative,5240
positive,1164
neutral,1108


#Preprocessing

In [ ]:
#DATA CLEANING
def cleaning_text(text):
    # Tahap-1: Menghapus karakter non-ascii (emoji, dll)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Tahap-2: Menghapus URLs (termasuk pic.twitter.com)
    text = re.sub(r'http[s]?\:\/\/[a-zA-Z0-9\.\/\_?=%&#\-\+!]+', ' ', text)
    text = re.sub(r'pic.twitter.com?[a-zA-Z0-9\.\/\_?=%&#\-\+!]+', ' ', text)
    # Tahap-3: Menghapus mention (@username)
    text = re.sub(r'\@([\w]+)', ' ', text)
    # Tahap-4: Kmenghapus tagar/hashtag (#trending)
    text = re.sub(r'\#([\w]+)',' ', text)
    # Tahap-5: Menghapus simbol dan tanda baca
    text = re.sub(r'[!$%^&*@#()_+|~=`{}\[\]%\-:";\'<>?,.\/]', ' ', text)
    # Tahap-6: Menghapus angka
    text = re.sub(r'[0-9]+', '', text)
    # Tahap-7: Koreksi duplikasi tiga karakter beruntun atau lebih (contoh: "haaa" menjadi "ha")
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text)
    # Tahap-8: Menghapus baris baru
    text = re.sub(r'\n', ' ', text)
    # Tahap-9: Menghapus spasi ganda atau lebih
    text = re.sub(r' +', ' ', text)
    # Tahap-10: spasi di awal dan akhir kalimat
    text = re.sub(r'^[ ]|[ ]$','', text)
    # Tahap-11: Konversi ke karakter huruf kecil
    text = text.lower()
    return text
# Melakukan pembersihan data
data_train['cleaned_text'] = data_train['text'].apply(cleaning_text)
data_val['cleaned_text'] = data_val['text'].apply(cleaning_text)
data_test['cleaned_text'] = data_test['text'].apply(cleaning_text)
data_train[['text', 'cleaned_text']].head()

,text,cleaned_text
0,Tanah dalam level perklenikan sudah pada level...,tanah dalam level perklenikan sudah pada level...
1,Tinggal menunggu kasus korupsi IKN,tinggal menunggu kasus korupsi ikn
2,Benar apa yg dikatakan politikus \n@PDI_Perjua...,benar apa yg dikatakan politikus sulit mencari...
3,Ada apa ya dibalik urgensi IKN?,ada apa ya dibalik urgensi ikn
4,Sudah tau akan jadi proyek mangkrak tapi ngoto...,sudah tau akan jadi proyek mangkrak tapi ngoto...


In [ ]:
#TOKENIZATION
# Download tokenizer untuk pertama kali
nltk.download('punkt_tab')

# Lakukan tokenisasi pada teks
data_train['tokenized_text'] = data_train['cleaned_text'].apply(word_tokenize)
data_val['tokenized_text'] = data_val['cleaned_text'].apply(word_tokenize)
data_test['tokenized_text'] = data_test['cleaned_text'].apply(word_tokenize)

# Tampilkan hasil tokenisasi
data_train[['cleaned_text', 'tokenized_text']].head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,cleaned_text,tokenized_text
0,tanah dalam level perklenikan sudah pada level...,"[tanah, dalam, level, perklenikan, sudah, pada..."
1,tinggal menunggu kasus korupsi ikn,"[tinggal, menunggu, kasus, korupsi, ikn]"
2,benar apa yg dikatakan politikus sulit mencari...,"[benar, apa, yg, dikatakan, politikus, sulit, ..."
3,ada apa ya dibalik urgensi ikn,"[ada, apa, ya, dibalik, urgensi, ikn]"
4,sudah tau akan jadi proyek mangkrak tapi ngoto...,"[sudah, tau, akan, jadi, proyek, mangkrak, tap..."


In [ ]:
#INDONESIAN SLANG & STOP WORDS REMOVAL
#ENGLISH WORDS REMOVAL
from google.colab import files
#upload file kamus stopword
stopword_file = files.upload()
#upload file kamus slang
slang_file = files.upload()

In [ ]:
# Memuat dataset custom slang
slang_df = pd.read_csv('/content/colloquial-indonesian-lexicon.csv')
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))  # Create a dictionary mapping slang to formal words
# Memuat dataset custom stopword
with open('/content/id.stopwords.02.01.2016 custom.txt', 'r', encoding='utf-8') as f:
    indonesia_stopwords = f.read().splitlines()
# Memuat dataset corpus english
nltk.download('words')
english_words = set(words.words())
# Fungsi untuk menghapus kata slangwords
def remove_slangwords(words):
    return [word for word in words if word.lower() not in slang_dict]
# Fungsi untuk menghapus stopwords
def remove_stopwords(words):
    return [word for word in words if word.lower() not in indonesia_stopwords]
# Fungsi untuk menghapus kata-kata berbahasa Inggris
def remove_english_words(words):
    return [word for word in words if word.lower() not in english_words]

# menerapkan penghapusan english slang dan stopword
data_train['removed_text'] = data_train['tokenized_text'].apply(remove_slangwords).apply(remove_stopwords).apply(remove_english_words)
data_val['removed_text'] = data_val['tokenized_text'].apply(remove_slangwords).apply(remove_stopwords).apply(remove_english_words)
data_test['removed_text'] = data_test['tokenized_text'].apply(remove_slangwords).apply(remove_stopwords).apply(remove_english_words)
data_train[['tokenized_text', 'removed_text']].head()

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


,tokenized_text,removed_text
0,"[tanah, dalam, level, perklenikan, sudah, pada...","[tanah, perklenikan, tertinggi, investasi, gag..."
1,"[tinggal, menunggu, kasus, korupsi, ikn]","[tinggal, menunggu, korupsi, ikn]"
2,"[benar, apa, yg, dikatakan, politikus, sulit, ...","[politikus, sulit, mencari, sekualitas, jokowi..."
3,"[ada, apa, ya, dibalik, urgensi, ikn]","[dibalik, urgensi, ikn]"
4,"[sudah, tau, akan, jadi, proyek, mangkrak, tap...","[proyek, mangkrak, ngotot, jaminan, ikn, dilan..."


In [ ]:
# STEMMING
# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi untuk melakukan stemming
def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

# Terapkan fungsi stemming pada kolom 'cleaned_text'
data_train['stemmed_text'] = data_train['removed_text'].apply(stemming)
data_val['stemmed_text'] = data_val['removed_text'].apply(stemming)
data_test['stemmed_text'] = data_test['removed_text'].apply(stemming)

# Tampilkan hasil
data_train[['removed_text', 'stemmed_text']].head()

,removed_text,stemmed_text
0,"[tanah, perklenikan, tertinggi, investasi, gag...","[tanah, klenik, tinggi, investasi, gagal, biar..."
1,"[tinggal, menunggu, korupsi, ikn]","[tinggal, tunggu, korupsi, ikn]"
2,"[politikus, sulit, mencari, sekualitas, jokowi...","[politikus, sulit, cari, kualitas, jokowi, san..."
3,"[dibalik, urgensi, ikn]","[balik, urgensi, ikn]"
4,"[proyek, mangkrak, ngotot, jaminan, ikn, dilan...","[proyek, mangkrak, ngotot, jamin, ikn, lanjut]"


In [ ]:
data_train

,text,label,cleaned_text,tokenized_text,removed_text,stemmed_text
0,Tanah dalam level perklenikan sudah pada level...,negative,tanah dalam level perklenikan sudah pada level...,"[tanah, dalam, level, perklenikan, sudah, pada...","[tanah, perklenikan, tertinggi, investasi, gag...","[tanah, klenik, tinggi, investasi, gagal, biar..."
1,Tinggal menunggu kasus korupsi IKN,negative,tinggal menunggu kasus korupsi ikn,"[tinggal, menunggu, kasus, korupsi, ikn]","[tinggal, menunggu, korupsi, ikn]","[tinggal, tunggu, korupsi, ikn]"
2,Benar apa yg dikatakan politikus \n@PDI_Perjua...,negative,benar apa yg dikatakan politikus sulit mencari...,"[benar, apa, yg, dikatakan, politikus, sulit, ...","[politikus, sulit, mencari, sekualitas, jokowi...","[politikus, sulit, cari, kualitas, jokowi, san..."
3,Ada apa ya dibalik urgensi IKN?,negative,ada apa ya dibalik urgensi ikn,"[ada, apa, ya, dibalik, urgensi, ikn]","[dibalik, urgensi, ikn]","[balik, urgensi, ikn]"
4,Sudah tau akan jadi proyek mangkrak tapi ngoto...,negative,sudah tau akan jadi proyek mangkrak tapi ngoto...,"[sudah, tau, akan, jadi, proyek, mangkrak, tap...","[proyek, mangkrak, ngotot, jaminan, ikn, dilan...","[proyek, mangkrak, ngotot, jamin, ikn, lanjut]"
...,...,...,...,...,...,...
3319,TOP..\n\nSemoga IKN bisa terbangun tepat waktu..,positive,top semoga ikn bisa terbangun tepat waktu,"[top, semoga, ikn, bisa, terbangun, tepat, waktu]","[semoga, ikn, terbangun]","[moga, ikn, bangun]"
3320,Wajar Jokowi geram grup WA TNI bahas tolak IKN...,positive,wajar jokowi geram grup wa tni bahas tolak ikn...,"[wajar, jokowi, geram, grup, wa, tni, bahas, t...","[wajar, jokowi, geram, grup, tni, bahas, tolak...","[wajar, jokowi, geram, grup, tni, bahas, tolak..."
3321,"Mantap banget dech , so pasti dong dukung IKN ...",positive,mantap banget dech so pasti dong dukung ikn nu...,"[mantap, banget, dech, so, pasti, dong, dukung...","[mantap, banget, dukung, ikn, nusantara]","[mantap, banget, dukung, ikn, nusantara]"
3322,Pindah IKN biasa aja kaleee... banyak negara m...,positive,pindah ikn biasa aja kale banyak negara melaku...,"[pindah, ikn, biasa, aja, kale, banyak, negara...","[pindah, ikn, negara, melakukannya, hubungkan]","[pindah, ikn, negara, laku, hubung]"


In [ ]:
#DATA UNUSED COLUMN REMOVAL
#Menhapus kolom yang tidak terpakai
data_train = data_train[['stemmed_text', 'label']]
data_val = data_val[['stemmed_text', 'label']]
data_test = data_test[['stemmed_text', 'label']]

data_train = data_train.rename(columns={'stemmed_text': 'text'})
data_val = data_val.rename(columns={'stemmed_text': 'text'})
data_test = data_test.rename(columns={'stemmed_text': 'text'})

data_train = data_train.rename(columns={'label': 'sentiment'})
data_val = data_val.rename(columns={'label': 'sentiment'})
data_test = data_test.rename(columns={'label': 'sentiment'})

data_train['text'] = data_train['text'].apply(lambda x: ' '.join(x))
data_val['text'] = data_val['text'].apply(lambda x: ' '.join(x))
data_test['text'] = data_test['text'].apply(lambda x: ' '.join(x))
data_train

,text,sentiment
0,tanah klenik tinggi investasi gagal biar dukun...,negative
1,tinggal tunggu korupsi ikn,negative
2,politikus sulit cari kualitas jokowi sanggup t...,negative
3,balik urgensi ikn,negative
4,proyek mangkrak ngotot jamin ikn lanjut,negative
...,...,...
3319,moga ikn bangun,positive
3320,wajar jokowi geram grup tni bahas tolak ikn pr...,positive
3321,mantap banget dukung ikn nusantara,positive
3322,pindah ikn negara laku hubung,positive


In [ ]:
#DATA SAVING TO TSV
data_train.to_csv('final_ikn_train_preprocessed.tsv', sep='\t', index=False, header=None)
data_val.to_csv('final_ikn_val_preprocessed.tsv', sep='\t', index=False, header=None)
data_test.to_csv('final_ikn_test_preprocessed.tsv', sep='\t', index=False, header=None)